# Advanced Registration Techniques

This notebook demonstrates advanced features of Space-map including:
- Working with real CSV data
- Different alignment methods
- Parameter tuning
- Quality assessment
- Handling large datasets

## Import Libraries

In [ ]:
import spacemap
from spacemap import Slice
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"Space-map version: {spacemap.__version__}")

## Example 1: Loading Real Data from CSV

Demonstrates how to load and process real spatial transcriptomics or proteomics data.

In [ ]:
# Load data from CSV
# Expected format: columns should include 'x', 'y', and 'layer' (or similar identifier)
# data_path = "path/to/your/cells.csv.gz"  # Replace with your data path

# For demonstration, create sample data
# In real usage, uncomment the line above and load your actual data
# df = pd.read_csv(data_path)

# Create sample data
np.random.seed(123)
sample_data = []
for layer in range(10):  # 10 layers
    n_cells = np.random.randint(800, 1200)
    for _ in range(n_cells):
        # Add realistic tissue structure
        if np.random.random() < 0.7:
            # Clustered cells
            cluster_center = np.random.choice([300, 500, 700], 2)
            x = np.random.normal(cluster_center[0], 80)
            y = np.random.normal(cluster_center[1], 80)
        else:
            # Scattered cells
            x = np.random.uniform(100, 900)
            y = np.random.uniform(100, 900)
        
        sample_data.append({
            'x': x,
            'y': y,
            'layer': layer,
            'cell_type': np.random.choice(['TypeA', 'TypeB', 'TypeC'])
        })

df = pd.DataFrame(sample_data)

print(f"Dataset summary:")
print(f"  Total cells: {len(df):,}")
print(f"  Layers: {df['layer'].nunique()}")
print(f"  Cell types: {df['cell_type'].unique().tolist()}")
print(f"\nCells per layer:")
print(df['layer'].value_counts().sort_index())

## Example 2: Data Preprocessing and Quality Check

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, layer in enumerate(sorted(df['layer'].unique())):
    layer_df = df[df['layer'] == layer]
    axes[i].scatter(layer_df['x'], layer_df['y'], s=0.5, alpha=0.5)
    axes[i].set_title(f'Layer {layer} ({len(layer_df)} cells)')
    axes[i].set_aspect('equal')

plt.tight_layout()
plt.show()

# Check for outliers
print("\nData range per layer:")
for layer in sorted(df['layer'].unique()):
    layer_df = df[df['layer'] == layer]
    print(f"  Layer {layer}: X[{layer_df['x'].min():.1f}, {layer_df['x'].max():.1f}], "
          f"Y[{layer_df['y'].min():.1f}, {layer_df['y'].max():.1f}]")

## Example 3: Initialize Project with Custom Parameters

In [ ]:
# Organize data by layers
xys = []
layer_ids = []

for layer in sorted(df['layer'].unique()):
    layer_data = df[df['layer'] == layer]
    xy = layer_data[['x', 'y']].values
    xys.append(xy)
    layer_ids.append(f"section_{layer:02d}")  # Custom layer naming

# Initialize project
BASE = "data/advanced_demo"
flowImport = spacemap.flow.FlowImport(BASE)

# You can customize the ratio parameter
flowImport.ratio = 1.5  # Adjust canvas size (default: 1.4)

# Initialize slices
flowImport.init_xys(xys, ids=layer_ids)
slices = flowImport.slices

print(f"Initialized project: {BASE}")
print(f"  Slices: {len(slices)}")
print(f"  XYRANGE: {spacemap.XYRANGE}")
print(f"  XYD: {spacemap.XYD}")

## Example 4: Different Alignment Methods

Space-map supports multiple alignment methods:
- `"auto"`: Automatically select best method
- `"sift"`: Traditional SIFT features
- `"sift_vgg"`: SIFT with VGG features
- `"loftr"`: LoFTR (Learning-based method)

In [ ]:
# Try different alignment methods
# Method 1: Auto (recommended for most cases)
mgr = spacemap.flow.AutoFlowMultiCenter4(slices, Slice.rawKey)
mgr.alignMethod = "auto"

# For SIFT-based alignment
# mgr.alignMethod = "sift"

# For deep learning-based alignment (requires more memory)
# mgr.alignMethod = "loftr"
# spacemap.affine_block.AutoAffineImgKey.useLDM = True

print(f"Using alignment method: {mgr.alignMethod}")

# Perform affine registration
mgr.affine("DF", show=False)
print("Affine registration completed!")

## Example 5: Fine Registration with LDDMM

In [ ]:
# Perform LDDMM registration
# This step uses GPU acceleration if available
mgr.ldm_pair(Slice.align1Key, Slice.align2Key, show=False)

print("LDDMM registration completed!")
print(f"Device used: {mgr.gpu if mgr.gpu else 'CPU'}")

## Example 6: Quality Assessment

Evaluate registration quality by comparing before and after alignment.

In [ ]:
# Compare alignment quality
fig, axes = plt.subplots(2, len(slices)-1, figsize=(4*(len(slices)-1), 8))
if len(slices) == 2:
    axes = axes.reshape(-1, 1)

for i in range(len(slices) - 1):
    # Before alignment (raw)
    points1_raw = slices[i].imgs[Slice.rawKey].get_points(Slice.rawKey)
    points2_raw = slices[i+1].imgs[Slice.rawKey].get_points(Slice.rawKey)
    
    # After alignment
    points1_aligned = slices[i].imgs[Slice.rawKey].get_points(Slice.align2Key)
    points2_aligned = slices[i+1].imgs[Slice.rawKey].get_points(Slice.align2Key)
    
    # Plot before
    axes[0, i].scatter(points1_raw[:, 0], points1_raw[:, 1], s=0.5, alpha=0.3, c='red')
    axes[0, i].scatter(points2_raw[:, 0], points2_raw[:, 1], s=0.5, alpha=0.3, c='blue')
    axes[0, i].set_title(f'Before: Layers {i} & {i+1}')
    axes[0, i].axis('equal')
    
    # Plot after
    axes[1, i].scatter(points1_aligned[:, 0], points1_aligned[:, 1], s=0.5, alpha=0.3, c='red')
    axes[1, i].scatter(points2_aligned[:, 0], points2_aligned[:, 1], s=0.5, alpha=0.3, c='blue')
    axes[1, i].set_title(f'After: Layers {i} & {i+1}')
    axes[1, i].axis('equal')

plt.tight_layout()
plt.savefig(f'{BASE}/alignment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Example 7: Export with Metadata

Export aligned coordinates along with cell type and other metadata.

In [ ]:
# Export with metadata preservation
export = spacemap.flow.FlowExport(slices)

# Collect aligned points with metadata
export_data = []
z_spacing = 10

for i, slice_obj in enumerate(slices):
    # Get aligned points
    points_aligned = slice_obj.imgs[Slice.rawKey].get_points(Slice.align2Key)
    
    # Get original layer data for metadata
    layer_id = slice_obj.id
    layer_num = int(layer_id.split('_')[1])
    layer_df = df[df['layer'] == layer_num].copy()
    
    # Match cells (assuming same order)
    if len(points_aligned) == len(layer_df):
        layer_df['x_aligned'] = points_aligned[:, 0]
        layer_df['y_aligned'] = points_aligned[:, 1]
        layer_df['z'] = i * z_spacing
        export_data.append(layer_df)

# Combine all layers
export_df = pd.concat(export_data, ignore_index=True)

# Save
export_df.to_csv(f'{BASE}/aligned_cells_with_metadata.csv.gz', 
                 index=False, compression='gzip')

print(f"Exported {len(export_df):,} cells with metadata")
print(f"Columns: {export_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(export_df.head())

## Example 8: 3D Visualization by Cell Type

In [ ]:
# Create 3D visualization colored by cell type
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Color by cell type
cell_types = export_df['cell_type'].unique()
colors = plt.cm.Set2(np.linspace(0, 1, len(cell_types)))
color_map = dict(zip(cell_types, colors))

for cell_type in cell_types:
    subset = export_df[export_df['cell_type'] == cell_type]
    ax.scatter(subset['x_aligned'], subset['y_aligned'], subset['z'],
               c=[color_map[cell_type]], s=1, alpha=0.5, label=cell_type)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z (Layer)')
ax.legend()
ax.set_title('3D Tissue Reconstruction by Cell Type')

plt.savefig(f'{BASE}/reconstruction_3d_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

## Example 9: Analyzing Cell Type Distribution Across Layers

In [ ]:
# Analyze cell type distribution
cell_type_dist = export_df.groupby(['layer', 'cell_type']).size().unstack(fill_value=0)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Stacked bar chart
cell_type_dist.plot(kind='bar', stacked=True, ax=ax1, colormap='Set2')
ax1.set_xlabel('Layer')
ax1.set_ylabel('Number of Cells')
ax1.set_title('Cell Type Distribution Across Layers')
ax1.legend(title='Cell Type')

# Proportion chart
cell_type_prop = cell_type_dist.div(cell_type_dist.sum(axis=1), axis=0)
cell_type_prop.plot(kind='bar', stacked=True, ax=ax2, colormap='Set2')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Proportion')
ax2.set_title('Cell Type Proportions Across Layers')
ax2.legend(title='Cell Type')

plt.tight_layout()
plt.savefig(f'{BASE}/cell_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCell type counts:")
print(cell_type_dist)

## Summary

In this advanced tutorial, we covered:
1. Loading and preprocessing real data
2. Data quality checking
3. Custom project initialization
4. Different alignment methods
5. Registration quality assessment
6. Exporting with metadata
7. Advanced 3D visualizations
8. Analyzing spatial patterns

## Tips for Real Data

- **Large datasets**: Consider processing subsets or using GPU acceleration
- **Memory issues**: Reduce `spacemap.XYD` for lower resolution processing
- **Alignment issues**: Try different alignment methods or adjust parameters
- **Quality check**: Always visualize intermediate results

## Next Steps

- See `03_image_features.ipynb` for incorporating histology images
- Check the [API documentation](https://a12910.github.io/space-map) for detailed parameter descriptions